In [ ]:
# Setup — run once per session
!pip install -q timm einops ml-collections medpy SimpleITK tensorboardX thop

!cp -r /kaggle/input/datasets/deepsotaai/adada-transunet-code/Ada-DA-TransUNet /kaggle/working/Ada-DA-TransUNet
!cp -r /kaggle/input/datasets/deepsotaai/vit-pretrained-weights/model            /kaggle/working/model

# Kaggle strips '+' from filenames — rename back (|| true silences error if already correct)
!mv /kaggle/working/model/vit_checkpoint/imagenet21k/R50ViT-B_16.npz \
    /kaggle/working/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz 2>/dev/null || true

# Prevent HuggingFace 'datasets' library from shadowing local datasets/ folder
!touch /kaggle/working/Ada-DA-TransUNet/datasets/__init__.py

# Symlink Synapse data (train.py hardcodes ../data/Synapse/ and ignores --root_path)
!mkdir -p /kaggle/working/data/Synapse
!ln -sfn /kaggle/input/datasets/dogcdt/synapse/Synapse/train_npz  /kaggle/working/data/Synapse/train_npz
!ln -sfn /kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5 /kaggle/working/data/Synapse/test_vol_h5

print('Setup complete.')

In [ ]:
import os, shutil, glob
import kagglehub

# ── Stage AdaDA checkpoint (from adada-transunet-checkpoints dataset) ──
ckpt_root = kagglehub.dataset_download('deepsotaai/adada-transunet-checkpoints')
print("Downloaded to:", ckpt_root)
print("Contents:", glob.glob(ckpt_root + '/**', recursive=True))

MODEL_DIR = '/kaggle/working/model/AdaDA_Synapse224/AdaDA_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224'
os.makedirs(MODEL_DIR, exist_ok=True)

src = os.path.join(ckpt_root, 'best_model.pth')
if os.path.exists(src):
    shutil.copy(src, os.path.join(MODEL_DIR, 'best_model.pth'))
    print("Staged: best_model.pth")
else:
    print("ERROR: best_model.pth not found in", ckpt_root)

In [ ]:
%%bash
echo "========================================"
echo " AdaDA-TransUNet  |  TRAINING"
echo " Started: $(date)"
echo "========================================"
cd /kaggle/working/Ada-DA-TransUNet
python -u train.py \
  --dataset      Synapse \
  --vit_name     R50-ViT-B_16 \
  --max_epochs   150 \
  --batch_size   24 \
  --base_lr      0.01 \
  --n_skip       3 \
  --img_size     224 \
  --window_size  7 \
  --rank         32 \
  --groups       8 \
  --seed         1234 \
  --val_interval 10
echo "========================================"
echo " Training finished: $(date)"
echo "========================================"

In [ ]:
%%bash
echo "========================================"
echo " AdaDA-TransUNet  |  INFERENCE"
echo " Started: $(date)"
echo "========================================"
cd /kaggle/working/Ada-DA-TransUNet
python -u test.py \
  --dataset      Synapse \
  --vit_name     R50-ViT-B_16 \
  --num_classes  9 \
  --img_size     224 \
  --window_size  7 \
  --rank         32 \
  --groups       8 \
  --is_savenii
echo "========================================"
echo " Inference finished: $(date)"
echo "========================================"